In [0]:
SHOW TABLES IN workspace.kraf_bronze;

In [0]:
CREATE SCHEMA IF NOT EXISTS workspace.kraf_silver;

In [0]:
CREATE OR REPLACE TABLE workspace.kraf_silver.sovereign_credit_ratings_silver AS

SELECT DISTINCT
    TRIM(Sovereign) AS sovereign,
    TRIM(Rating_foreign) AS rating_foreign,
    TRIM(Rating_foreign_outlook) AS rating_foreign_outlook,
    TRIM(Rating_local) AS rating_local,
    TRIM(Rating_local_outlook) AS rating_local_outlook,
    TRIM(Country_ceilings_foreign) AS country_ceilings_foreign,
    TRIM(Country_ceilings_local) AS country_ceilings_local

FROM workspace.kraf_bronze.sovereign_credit_ratings_bronze

WHERE Sovereign IS NOT NULL
  AND TRIM(Sovereign) <> '';

In [0]:
DESCRIBE workspace.kraf_bronze.sovereign_credit_ratings_bronze;

In [0]:
SELECT COUNT(*) AS total_registros
FROM workspace.kraf_silver.sovereign_credit_ratings_silver;

In [0]:
SELECT *
FROM workspace.kraf_silver.sovereign_credit_ratings_silver
LIMIT 10;

In [0]:
CREATE OR REPLACE TABLE workspace.kraf_silver.global_inflation_silver AS

SELECT DISTINCT
    TRIM(`country_code`) AS country_code,
    TRIM(`country_name`) AS country_name,
    TRIM(`region`) AS region,
    TRIM(`sub_region`) AS sub_region,
    TRIM(`intermediate_region`) AS intermediate_region,
    TRIM(`indicator_code`) AS indicator_code,
    TRIM(`indicator_name`) AS indicator_name,
    CAST(`year` AS INT) AS year,
    CAST(`inflation_rate` AS DOUBLE) AS inflation_rate

FROM workspace.kraf_bronze.global_inflation_bronze

WHERE `country_name` IS NOT NULL
  AND TRIM(`country_name`) <> ''
  AND `year` IS NOT NULL
  AND `inflation_rate` IS NOT NULL;

In [0]:
SELECT COUNT(*) AS total_registros
FROM workspace.kraf_silver.global_inflation_silver;

In [0]:
SELECT *
FROM workspace.kraf_silver.global_inflation_silver
ORDER BY year DESC
LIMIT 10;

In [0]:
DESCRIBE workspace.kraf_bronze.currency_exchange_rates_bronze;

In [0]:
CREATE OR REPLACE TABLE workspace.kraf_silver.currency_exchange_rates_silver AS

SELECT DISTINCT
    `Date` AS fecha,
    `Brazilian Real` AS brazilian_real,
    `Chilean Peso` AS chilean_peso,
    `Colombian Peso` AS colombian_peso,
    `Mexican Peso` AS mexican_peso

FROM workspace.kraf_bronze.currency_exchange_rates_bronze

WHERE `Date` IS NOT NULL;

In [0]:
SELECT COUNT(*) AS total_registros
FROM workspace.kraf_silver.currency_exchange_rates_silver;

In [0]:
SELECT *
FROM workspace.kraf_silver.currency_exchange_rates_silver
ORDER BY fecha DESC
LIMIT 10;

In [0]:
DESCRIBE workspace.kraf_bronze.nemotron_personas_sv_bronze;

In [0]:
SHOW COLUMNS IN workspace.kraf_bronze.nemotron_personas_sv_bronze;

In [0]:
SELECT
    ordinal_position,
    column_name,
    data_type
FROM workspace.information_schema.columns
WHERE table_schema = 'kraf_bronze'
  AND table_name = 'nemotron_personas_sv_bronze'
ORDER BY ordinal_position;

In [0]:
SELECT
    ordinal_position,
    column_name,
    data_type
FROM workspace.information_schema.columns
WHERE table_schema = 'kraf_bronze'
  AND table_name = 'nemotron_personas_sv_bronze'
  AND ordinal_position >= 15
ORDER BY ordinal_position;

In [0]:
CREATE OR REPLACE TABLE workspace.kraf_silver.nemotron_personas_sv_silver AS

SELECT DISTINCT
    TRIM(`uuid`) AS uuid,
    CAST(`age` AS INT) AS age,
    TRIM(`sex`) AS sex,
    TRIM(`education_level`) AS education_level,
    TRIM(`occupation`) AS occupation,
    TRIM(`area`) AS area,
    TRIM(`municipality`) AS municipality,
    TRIM(`department`) AS department,
    TRIM(`country`) AS country,
    TRIM(`marital_status`) AS marital_status,
    TRIM(`household_type`) AS household_type,
    TRIM(`languages_spoken`) AS languages_spoken

FROM workspace.kraf_bronze.nemotron_personas_sv_bronze

WHERE `uuid` IS NOT NULL
  AND `age` IS NOT NULL
  AND `country` IS NOT NULL
  AND TRIM(`country`) <> '';

In [0]:
SELECT COUNT(*) AS total_registros
FROM workspace.kraf_silver.nemotron_personas_sv_silver;

In [0]:
SELECT *
FROM workspace.kraf_silver.nemotron_personas_sv_silver
LIMIT 10;

In [0]:
SHOW TABLES IN workspace.kraf_silver;

In [0]:
SELECT 'transferencia_silver' AS tabla, COUNT(*) AS registros FROM workspace.kraf_silver.transferencia_silver
UNION ALL
SELECT 'global_inflation_silver', COUNT(*) FROM workspace.kraf_silver.global_inflation_silver
UNION ALL
SELECT 'sovereign_credit_ratings_silver', COUNT(*) FROM workspace.kraf_silver.sovereign_credit_ratings_silver
UNION ALL
SELECT 'currency_exchange_rates_silver', COUNT(*) FROM workspace.kraf_silver.currency_exchange_rates_silver
UNION ALL
SELECT 'nemotron_personas_sv_silver', COUNT(*) FROM workspace.kraf_silver.nemotron_personas_sv_silver;

In [0]:
-- ============================================================
-- KRAF | FASE 2 | SILVER
-- Tabla: saldo_historico_silver
-- ============================================================

CREATE OR REPLACE TABLE workspace.kraf_silver.saldo_historico_silver AS
SELECT
    saldo_id,
    cuenta_id,
    fecha,
    CAST(saldo_inicial AS DECIMAL(15,2)) AS saldo_inicial,
    CAST(saldo_final AS DECIMAL(15,2)) AS saldo_final,
    CAST(saldo_final - saldo_inicial AS DECIMAL(15,2)) AS variacion_saldo,
    _ingested_at,
    _source
FROM workspace.kraf_bronze.saldo_historico_bronze
WHERE saldo_id IS NOT NULL
  AND cuenta_id IS NOT NULL
  AND fecha IS NOT NULL;

In [0]:
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT saldo_id) AS ids_unicos,
    SUM(CASE WHEN variacion_saldo IS NULL THEN 1 ELSE 0 END) AS variaciones_nulas
FROM workspace.kraf_silver.saldo_historico_silver;

In [0]:
SELECT
    saldo_id,
    cuenta_id,
    fecha,
    saldo_inicial,
    saldo_final,
    variacion_saldo
FROM workspace.kraf_silver.saldo_historico_silver
ORDER BY saldo_id
LIMIT 10;
